# Chapter 2: Computational Biology Foundations for AI

## Hands-On Jupyter Notebook

This notebook builds practical intuition for computational oncology by simulating a synthetic cohort and walking through mutation prevalence, expression analysis, statistical testing, and pathway-level interpretation.

**Estimated Time**: 60-90 minutes

**What you will practice**

- Constructing a biologically plausible synthetic cohort
- Quantifying genomic alteration prevalence
- Running differential expression style comparisons with FDR correction
- Aggregating signals at a pathway level
- Translating results into evidence-aware summaries

---

## Section 1: Setup and Imports

We import standard scientific Python tooling and set plotting defaults.

In [1]:
import warnings
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

RNG = np.random.default_rng(42)

print("Environment configured successfully")

Environment configured successfully


## Section 2: Build a Synthetic Cancer Cohort

To keep the notebook reproducible and safe to share, we generate synthetic patient-level data with mutation, expression, and metadata fields.

In [ ]:
n_samples = 240
sample_ids = [f"sample_{i:03d}" for i in range(n_samples)]

tumor_types = RNG.choice(
    ["LUAD", "BRCA", "CRC", "MEL"],
    size=n_samples,
    p=[0.35, 0.3, 0.2, 0.15],
)

stages = RNG.choice(
    ["I", "II", "III", "IV"],
    size=n_samples,
    p=[0.2, 0.35, 0.3, 0.15],
)

base_mutation_probs: Dict[str, Dict[str, float]] = {
    "LUAD": {"EGFR": 0.24, "KRAS": 0.30, "TP53": 0.45, "PIK3CA": 0.12},
    "BRCA": {"EGFR": 0.05, "KRAS": 0.08, "TP53": 0.42, "PIK3CA": 0.28},
    "CRC": {"EGFR": 0.03, "KRAS": 0.38, "TP53": 0.52, "PIK3CA": 0.18},
    "MEL": {"EGFR": 0.04, "KRAS": 0.14, "TP53": 0.50, "PIK3CA": 0.10},
}

mut_rows: List[Dict[str, int]] = []
for disease in tumor_types:
    probs = base_mutation_probs[disease]
    mut_rows.append(
        {f"{gene}_mut": int(RNG.random() < prob) for gene, prob in probs.items()}
    )

mutation_df = pd.DataFrame(mut_rows)

risk_signal = (
    (stages == "IV").astype(int) * 1.2
    + mutation_df["TP53_mut"].to_numpy() * 0.7
    + mutation_df["KRAS_mut"].to_numpy() * 0.3
    + RNG.normal(0, 0.6, n_samples)
)
high_risk = (risk_signal > np.quantile(risk_signal, 0.65)).astype(int)

cohort_df = pd.DataFrame(
    {
        "sample_id": sample_ids,
        "tumor_type": tumor_types,
        "stage": stages,
        "high_risk": high_risk,
    }
).join(mutation_df)

genes = [
    "EGFR", "ERBB2", "KRAS", "BRAF", "TP53",
    "CDKN2A", "MKI67", "PDCD1", "IFNG", "VEGFA",
]

expr = pd.DataFrame(
    RNG.normal(loc=0.0, scale=1.0, size=(n_samples, len(genes))),
    columns=genes,
    index=sample_ids,
)

expr["MKI67"] += high_risk * 0.8 + (stages == "IV").astype(int) * 0.5
expr["VEGFA"] += (stages == "IV").astype(int) * 0.6
expr["EGFR"] += cohort_df["EGFR_mut"].to_numpy() * 1.0
expr["KRAS"] += cohort_df["KRAS_mut"].to_numpy() * 0.9
expr["PDCD1"] += (cohort_df["tumor_type"].eq("MEL").to_numpy()) * 0.6
expr["IFNG"] += (cohort_df["tumor_type"].eq("MEL").to_numpy()) * 0.4

expression_df = expr.reset_index().rename(columns={"index": "sample_id"})
analysis_df = cohort_df.merge(expression_df, on="sample_id", how="left")

print("Synthetic cohort built")
print("Shape:", analysis_df.shape)
analysis_df.head()

Synthetic cohort built
Shape: (240, 18)


,sample_id,tumor_type,stage,high_risk,EGFR_x,KRAS_x,TP53_x,PIK3CA,EGFR_y,ERBB2,KRAS_y,BRAF,TP53_y,CDKN2A,MKI67,PDCD1,IFNG,VEGFA
0,sample_000,CRC,III,1,0,1,0,0,-1.471115,0.503049,-1.514108,0.648394,-1.492106,0.053458,NaN,NaN,NaN,-0.694139
1,sample_001,BRCA,III,0,1,0,0,1,0.493397,1.272883,0.536297,-0.385630,-0.236446,-1.567177,NaN,NaN,NaN,0.128622
2,sample_002,MEL,II,0,0,0,0,0,2.386952,-1.100136,-0.102662,1.798288,1.191390,-0.696212,NaN,NaN,NaN,-0.698680
3,sample_003,CRC,II,0,0,1,0,0,-0.089320,-1.598052,1.035123,0.450021,-0.227419,-1.615051,NaN,NaN,NaN,-0.280600
4,sample_004,LUAD,III,0,0,0,1,0,-1.500669,-1.025410,-0.537811,1.228421,-0.639503,0.640214,NaN,NaN,NaN,0.873057


## Section 3: Statistical Analysis

We compute alteration prevalence and run a simple differential-expression style comparison between EGFR-mutant and EGFR-wildtype groups, then control for multiple testing using Benjamini-Hochberg FDR.

In [ ]:
def benjamini_hochberg(p_values: pd.Series) -> pd.Series:
    """Return FDR-adjusted p-values using Benjamini-Hochberg."""
    p = p_values.fillna(1.0).clip(0, 1).to_numpy()
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * m / (np.arange(1, m + 1))
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty_like(adjusted)
    out[order] = adjusted
    return pd.Series(out, index=p_values.index)

mutation_cols = ["EGFR_mut", "KRAS_mut", "TP53_mut", "PIK3CA_mut"]
mutation_prevalence = analysis_df[mutation_cols].mean().sort_values(ascending=False) * 100
mutation_prevalence.index = mutation_prevalence.index.str.replace("_mut", "", regex=False)

group_a = analysis_df[analysis_df["EGFR_mut"] == 1]
group_b = analysis_df[analysis_df["EGFR_mut"] == 0]

genes_for_test = ["ERBB2", "KRAS", "BRAF", "TP53", "CDKN2A", "MKI67", "PDCD1", "IFNG", "VEGFA"]
rows = []
for gene in genes_for_test:
    stat = ttest_ind(group_a[gene], group_b[gene], equal_var=False)
    rows.append(
        {
            "gene": gene,
            "mean_mut": group_a[gene].mean(),
            "mean_wt": group_b[gene].mean(),
            "effect_size": group_a[gene].mean() - group_b[gene].mean(),
            "p_value": stat.pvalue,
        }
    )

de_results = pd.DataFrame(rows).set_index("gene")
de_results["fdr"] = benjamini_hochberg(de_results["p_value"])
de_results = de_results.sort_values(["fdr", "effect_size"], ascending=[True, False])

print("Mutation prevalence (%)")
display(mutation_prevalence.to_frame("prevalence_percent"))

print("\nDifferential-expression style results (EGFR mutant vs wildtype)")
de_results

## Section 4: Pathway-Level Interpretation

Gene-level results are helpful, but pathway aggregation is often more stable and biologically interpretable.

In [ ]:
pathway_map = {
    "Proliferation": ["MKI67", "CDKN2A", "TP53"],
    "Immune_Response": ["PDCD1", "IFNG"],
    "Growth_Signaling": ["EGFR", "ERBB2", "KRAS", "BRAF"],
    "Angiogenesis": ["VEGFA"],
}

pathway_scores = pd.DataFrame({"sample_id": analysis_df["sample_id"]})
for pathway, pathway_genes in pathway_map.items():
    pathway_scores[pathway] = analysis_df[pathway_genes].mean(axis=1)

plot_df = pathway_scores.merge(analysis_df[["sample_id", "high_risk"]], on="sample_id")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mutation_prevalence.plot(kind="bar", ax=axes[0], color="#3B82F6")
axes[0].set_title("Driver Alteration Prevalence")
axes[0].set_ylabel("Percent of samples")
axes[0].set_xlabel("Gene")
axes[0].tick_params(axis="x", rotation=0)

sns.boxplot(
    data=plot_df.melt(id_vars=["sample_id", "high_risk"], value_vars=list(pathway_map.keys())),
    x="variable",
    y="value",
    hue="high_risk",
    ax=axes[1],
)
axes[1].set_title("Pathway Scores by Risk Group")
axes[1].set_xlabel("Pathway")
axes[1].set_ylabel("Average pathway score")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend(title="high_risk", labels=["0", "1"] )

plt.tight_layout()
plt.show()

pathway_summary = (
    plot_df.groupby("high_risk")[list(pathway_map.keys())].mean().T
    .rename(columns={0: "risk_0_mean", 1: "risk_1_mean"})
)
pathway_summary["difference_1_minus_0"] = pathway_summary["risk_1_mean"] - pathway_summary["risk_0_mean"]
pathway_summary.sort_values("difference_1_minus_0", ascending=False)

## Section 5: Exercises

### Exercise 1: Add a New Biomarker Comparison

Compare expression of the `PDCD1` gene between KRAS-mutant and KRAS-wildtype samples. Report group means, an effect size (difference in means), and a statistical p-value.

### Exercise 2: Define a Composite Pathway Score

Create a custom score for a pathway of your choice (for example, cell cycle) using 3-5 genes from the dataset, then compare scores by high-risk status.

In [ ]:
# Exercise 1 starter: PDCD1 comparison by KRAS mutation status
exercise_gene = "PDCD1"
mut_col = "KRAS_mut"

kras_mut = analysis_df[analysis_df[mut_col] == 1][exercise_gene]
kras_wt = analysis_df[analysis_df[mut_col] == 0][exercise_gene]

exercise1_summary = pd.DataFrame(
    {
        "group": ["KRAS_mut", "KRAS_wildtype"],
        "mean_expression": [kras_mut.mean(), kras_wt.mean()],
        "std_expression": [kras_mut.std(), kras_wt.std()],
        "n": [kras_mut.shape[0], kras_wt.shape[0]],
    }
)

exercise1_p = ttest_ind(kras_mut, kras_wt, equal_var=False).pvalue
exercise1_effect = kras_mut.mean() - kras_wt.mean()

print("Exercise 1 summary")
display(exercise1_summary)
print(f"Effect size (mut - wt): {exercise1_effect:.3f}")
print(f"p-value: {exercise1_p:.4g}")

### Exercise Notes

Keep your outputs grounded in uncertainty-aware reporting. For every claim, include at least one quantitative statistic and one caveat about synthetic data assumptions.

In [ ]:
# Exercise 2 starter: create a custom composite pathway score
custom_pathway_genes = ["MKI67", "TP53", "VEGFA"]

missing = [g for g in custom_pathway_genes if g not in analysis_df.columns]
if missing:
    raise ValueError(f"Missing genes in analysis_df: {missing}")

analysis_df["custom_pathway_score"] = analysis_df[custom_pathway_genes].mean(axis=1)

risk_group_summary = (
    analysis_df.groupby("high_risk")["custom_pathway_score"]
    .agg(["mean", "std", "count"])
)

score_test = ttest_ind(
    analysis_df.loc[analysis_df["high_risk"] == 1, "custom_pathway_score"],
    analysis_df.loc[analysis_df["high_risk"] == 0, "custom_pathway_score"],
    equal_var=False,
)

display(risk_group_summary)
print(f"Difference (risk=1 minus risk=0): {risk_group_summary.loc[1, 'mean'] - risk_group_summary.loc[0, 'mean']:.3f}")
print(f"p-value: {score_test.pvalue:.4g}")

## Section 6: Key Takeaways

- Multi-modal oncology data requires context-aware interpretation, not single-feature reasoning.
- Multiple-testing correction is essential in high-dimensional analyses.
- Pathway-level summaries can be more stable than individual genes.
- Computational conclusions should include effect size, uncertainty, and assumptions.

---

## Next Steps

1. Complete both exercises and compare your findings with peers.
2. Move to Chapter 3 to connect these concepts to real biomedical data ecosystems.
3. Revisit this notebook later when building retrieval and genomics agents.

---

**Last Updated**: May 2026